# D3 & D4 Research Overview: All Runs & Key Findings

**Date**: 2026-04-21 | **Scope**: D3 background + D4 FoundOpt all RL variations + Multi-goal B4 (6 goals)

**Opus Eval Scale**: 1-10 per dimension (total /40). Upgraded from earlier 1-5 scale (total /20).

---

**数据来源**:
1. FoundOpt 各 run 的 metrics: `projects/grant_proposal/runs/*/config.json`, `*/metrics.jsonl`
2. 标准化 Opus 评估 (8 RL 变体): `projects/grant_proposal/analysis/foundopt_all_opus_eval/opus_scores.jsonl`
3. 多目标 B4 Opus 评估: `projects/grant_proposal/analysis/wave1_opus_eval/opus_scores.jsonl`
4. 多 prompt 验证: `projects/grant_proposal/analysis/d4v7_foundopt_ablation/multi_prompt_validation.json`
5. 早期消融 Opus 评估: `projects/grant_proposal/analysis/d4v7_foundopt_ablation/data/opus_eval_results.jsonl`

**核心问题**: RL 训练能否超越纯搜索 (B4) 提升研究计划的真实质量 (Opus judge)?

In [13]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path("/home/silas/co-scientist-project")
RUNS = ROOT / "projects/grant_proposal/runs"
ANALYSIS = ROOT / "projects/grant_proposal/analysis"

# ── 1. FoundOpt run configs + final metrics ──────────────────────────────────
TARGET_RUNS = {
    "B4_orig":       "2026_04_20_d4v7_01_foundopt_B4",
    "MAIN_orig":     "2026_04_20_d4v7_01_foundopt_MAIN",
    "A_fresh":       "2026_04_20_d4v7_ablation_A_fresh",
    "CR_only":       "2026_04_20_d4v7_ablation_CR_only",
    "BR_MAIN_235B":  "2026_04_20_better_reward_MAIN",
    "BR_B4_235B":    "2026_04_20_better_reward_B4",
    "GAPO":          "2026_04_21_gapo_foundopt",
    "B4_paper":      "2026_04_21_b4_paper_01_foundopt",
}

configs, final_metrics = {}, {}
for tag, dirname in TARGET_RUNS.items():
    run_dir = RUNS / dirname
    with open(run_dir / "config.json") as f:
        configs[tag] = json.load(f)
    # 读最后一行 metrics
    with open(run_dir / "metrics.jsonl") as f:
        lines = f.readlines()
        final_metrics[tag] = json.loads(lines[-1])

# ── 2. Opus eval: 8 RL 变体 (standardized depth audit) ──────────────────────
opus_8rl_path = ANALYSIS / "foundopt_all_opus_eval/opus_scores.jsonl"
opus_8rl = []
with open(opus_8rl_path) as f:
    for line in f:
        opus_8rl.append(json.loads(line))
df_opus_8rl = pd.DataFrame(opus_8rl)

# ── 3. 多目标 B4 Opus eval ──────────────────────────────────────────────────
wave1_path = ANALYSIS / "wave1_opus_eval/opus_scores.jsonl"
wave1 = []
with open(wave1_path) as f:
    for line in f:
        wave1.append(json.loads(line))
df_wave1 = pd.DataFrame(wave1)

# ── 4. 多 prompt 验证 ───────────────────────────────────────────────────────
mpv_path = ANALYSIS / "d4v7_foundopt_ablation/multi_prompt_validation.json"
with open(mpv_path) as f:
    mpv_data = json.load(f)
df_mpv = pd.DataFrame(mpv_data)

# ── 5. 早期消融 Opus eval ───────────────────────────────────────────────────
ablation_opus_path = ANALYSIS / "d4v7_foundopt_ablation/data/opus_eval_results.jsonl"
ablation_opus = []
with open(ablation_opus_path) as f:
    for line in f:
        ablation_opus.append(json.loads(line))
df_ablation_opus = pd.DataFrame(ablation_opus)

print("=== 数据加载完成 ===")
print(f"目标 runs: {len(TARGET_RUNS)}")
print(f"Opus 8-RL 评估: {len(df_opus_8rl)} rows")
print(f"Wave1 多目标评估: {len(df_wave1)} rows")
print(f"多 prompt 验证: {len(df_mpv)} rows")
print(f"早期消融 Opus: {len(df_ablation_opus)} rows")

=== 数据加载完成 ===
目标 runs: 8
Opus 8-RL 评估: 8 rows
Wave1 多目标评估: 8 rows
多 prompt 验证: 9 rows
早期消融 Opus: 10 rows


## D3 背景: 为什么从 D3 转向 D4

D3 (TTT-Discover) 的核心发现推动了 D4 的设计:

### 1. CR-v7 架构效应: +0.19 来自架构而非 RL
- 早期消融 (MAIN vs B4) 显示, critique-revise pipeline 本身带来了 +0.19 的 Qwen 分数提升
- B4 (无 RL, 仅 in-context 搜索) 在 buffer_max 上等于甚至超过 MAIN (有 RL)
- **结论**: 架构 >> RL 梯度

### 2. 组件层级: critique >> RL >> nothing
- CR-v6 pilot: 0.545 -> 0.815 (+0.27), 其中大部分来自 critique-revise
- RL 的独特贡献: 探索 (fresh generation), 在 BoN=1 时 +0.087 Qwen, +2 Opus

### 3. v9 信号硬化: Opus 7-8 -> 9
- 从 v8.1 到 v9: 更严格的 signal prompts, 更好的 calibration
- 单一最好的 MAIN 计划从 Opus 7-8 提升到 9

### 4. Goodhart 天花板量化
- Qwen3-30B 给出的高分 (0.782-0.815) 计划在 Opus judge 上只得 7-15/40
- 参考方案在 Opus 上稳定 34-37/40
- Spearman rho = -0.16 (条件内)

**D4 的目标**: 在 D3 发现的基础上, 通过更强的 grader (235B) 和更好的信号设计, 打破这个天花板。

In [ ]:
# ── Cell 4: 综合 FoundOpt 运行表 ────────────────────────────────────────────
# 从 config.json + metrics.jsonl + opus_scores.jsonl 构建

rows = []
for tag in TARGET_RUNS:
    cfg = configs[tag]
    met = final_metrics[tag]

    # 模型简称
    policy = "235B" if "235B" in cfg["model_name"] else "30B"
    grader = "235B" if "235B" in cfg["grader_model_name"] else "30B"

    # RL config 描述
    skip_rl = cfg.get("skip_rl_update", False)
    train_fresh = cfg.get("train_on_fresh", False)
    skip_rev_rl = cfg.get("skip_revision_rl", False)
    diversity = cfg.get("diversity_method", "none")

    if skip_rl:
        rl_desc = "No RL (B4)"
    elif train_fresh and skip_rev_rl:
        if diversity == "gapo":
            rl_desc = "Fresh RL + GAPO diversity"
        else:
            rl_desc = "Fresh RL only"
    elif train_fresh and not skip_rev_rl:
        rl_desc = "Full RL (fresh + revise)"
    elif not train_fresh and not skip_rev_rl:
        rl_desc = "Revise RL only"
    else:
        rl_desc = "Custom"

    # Opus 分数匹配
    opus_row = df_opus_8rl[df_opus_8rl["tag"] == tag]
    opus_total = int(opus_row["total"].iloc[0]) if len(opus_row) > 0 else None

    iters_completed = met["iter"] + 1
    buf_max = met.get("reward/buffer_max", None)
    qwen_score = opus_row["qwen"].iloc[0] if len(opus_row) > 0 else met.get("reward/buffer_max")

    rows.append({
        "Tag": tag,
        "Policy": policy,
        "Grader": grader,
        "RL Config": rl_desc,
        "Iters": iters_completed,
        "buf_max": f"{buf_max:.3f}" if buf_max else "N/A",
        "Qwen Score": f"{qwen_score:.3f}" if qwen_score else "N/A",
        "Opus /40": opus_total,
    })

df_runs = pd.DataFrame(rows).sort_values("Opus /40", ascending=False).reset_index(drop=True)

print("=" * 90)
print("D4 FoundOpt 全部运行结果 (按 Opus 分数降序)")
print(f"数据来源: configs from {RUNS}, opus from {opus_8rl_path}")
print("评分标准: 1-10 per dimension, total /40")
print("=" * 90)
display(df_runs)

# 关键统计
grader_235b = df_runs[df_runs["Grader"] == "235B"]["Opus /40"]
grader_30b = df_runs[df_runs["Grader"] == "30B"]["Opus /40"]
print(f"\n235B grader 平均 Opus: {grader_235b.mean():.1f}/40 (n={len(grader_235b)})")
print(f"30B grader 平均 Opus:  {grader_30b.mean():.1f}/40 (n={len(grader_30b)})")
print(f"差异: +{grader_235b.mean() - grader_30b.mean():.1f} points")

In [ ]:
# ── Cell 5: Opus 分数柱状图 ──────────────────────────────────────────────────
# 按 Opus total 排序, 按 grader model 着色

df_plot = df_runs.sort_values("Opus /40", ascending=True).reset_index(drop=True)
colors = ["#2196F3" if g == "235B" else "#FF9800" for g in df_plot["Grader"]]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(df_plot)), df_plot["Opus /40"], color=colors, edgecolor="white", height=0.7)

# 标注
for i, (val, tag) in enumerate(zip(df_plot["Opus /40"], df_plot["Tag"])):
    ax.text(val + 0.3, i, f"{val}/40", va="center", fontsize=10, fontweight="bold")

ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels([f"{row['Tag']}\n({row['RL Config']})" for _, row in df_plot.iterrows()], fontsize=9)
ax.set_xlabel("Opus Total Score (/40)")
ax.set_title("D4 FoundOpt: 8 RL Variations — Opus Depth Audit Score (1-10 scale)\n(Blue=235B grader, Orange=30B grader)")
ax.set_xlim(0, 44)
ax.axvline(x=37, color="green", linestyle="--", alpha=0.5, label="Reference (37/40)")

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2196F3", label="Qwen3-235B policy+grader"),
    Patch(facecolor="#FF9800", label="Qwen3-30B policy+grader"),
    plt.Line2D([0], [0], color="green", linestyle="--", alpha=0.5, label="Reference (37/40)")
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

# 分维度明细
print("\n=== Opus 分维度分数 (1-10 per dimension, total /40) ===")
print(f"数据来源: {opus_8rl_path}")
detail_cols = ["tag", "depth", "methods", "feasibility", "grounding", "total"]
display(df_opus_8rl[detail_cols].sort_values("total", ascending=False).reset_index(drop=True))

## Finding 1: Grader/Policy 模型能力 >> RL 方法选择

**核心数据** (来自 `foundopt_all_opus_eval/opus_scores.jsonl`, 1-10 scale, total /40):

上表揭示了一个清晰的层级结构:

| 层级 | 条件 | Opus 范围 |
|---|---|---|
| **Tier 1** | 235B policy + 235B grader | 26-29/40 |
| **Tier 2** | 30B policy + 30B grader | 13-19/40 |
| **Reference** | 人类专家 | 34-37/40 |

**关键洞察**:
- 从 30B -> 235B 带来的 Opus 提升 (+12.0 points) **远大于**任何 RL 方法变体之间的差异 (30B 内部最大差异仅 6 points)
- 在同一 grader 内, RL 方法的选择 (full RL, fresh only, B4, GAPO) 影响很小
- **含义**: 模型能力是天花板, RL 是在天花板下的微调。优先投资更强的模型, 而非更复杂的 RL 算法

In [ ]:
# ── Cell 7: Qwen-Opus 相关性分析 ─────────────────────────────────────────────
# 数据来源: multi_prompt_validation.json (9 plans, 3 conditions x 3 quality tiers)

print(f"数据来源: {mpv_path}")
print(f"样本数: {len(df_mpv)} plans")
print("评分标准: 1-10 per dimension, total /40\n")

# ── 全局相关性 (跨条件) ──────────────────────────────────────────────────────
qwen_vals = df_mpv["qwen_agg"].values
opus_vals = df_mpv["opus"].values

rho_global, p_global = spearmanr(qwen_vals, opus_vals)
print(f"跨条件 Spearman rho: {rho_global:.3f} (p={p_global:.4f})")

# ── 条件内相关性 ─────────────────────────────────────────────────────────────
conditions = {"MAIN": [], "B4": [], "A_fresh": []}
for _, row in df_mpv.iterrows():
    for cond in conditions:
        if row["label"].startswith(cond):
            conditions[cond].append(row)

print("\n条件内 Spearman rho:")
for cond, cond_rows in conditions.items():
    cdf = pd.DataFrame(cond_rows)
    if len(cdf) >= 3:
        r, p = spearmanr(cdf["qwen_agg"], cdf["opus"])
        print(f"  {cond}: rho={r:.3f} (p={p:.4f}, n={len(cdf)})")

# ── 散点图 ───────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左: 跨条件散点图
cond_colors = {"MAIN": "#E53935", "B4": "#1E88E5", "A_fresh": "#43A047"}
for cond, cond_rows in conditions.items():
    cdf = pd.DataFrame(cond_rows)
    ax1.scatter(cdf["qwen_agg"], cdf["opus"], c=cond_colors[cond],
                label=cond, s=100, edgecolors="black", zorder=3)
    for _, row in cdf.iterrows():
        tier = row["label"].split("_")[-1]
        ax1.annotate(tier, (row["qwen_agg"], row["opus"]),
                    fontsize=7, ha="center", va="bottom", xytext=(0, 5),
                    textcoords="offset points")

ax1.set_xlabel("Qwen Aggregate Score")
ax1.set_ylabel("Opus Total (/40)")
ax1.set_title(f"跨条件 Qwen-Opus 相关性\nSpearman rho = {rho_global:.3f}")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 右: 8-run 对比 (foundopt_all_opus_eval)
qwen_8 = df_opus_8rl["qwen"].values
opus_8 = df_opus_8rl["total"].values
rho_8, p_8 = spearmanr(qwen_8, opus_8)

ax2.scatter(qwen_8, opus_8, c=["#2196F3" if "235B" in t else "#FF9800" for t in df_opus_8rl["tag"]],
            s=100, edgecolors="black", zorder=3)
for _, row in df_opus_8rl.iterrows():
    ax2.annotate(row["tag"], (row["qwen"], row["total"]),
                fontsize=7, ha="center", va="bottom", xytext=(0, 5),
                textcoords="offset points")

ax2.set_xlabel("Qwen buf_max Score")
ax2.set_ylabel("Opus Total (/40)")
ax2.set_title(f"8-Run Qwen-Opus 对比\nSpearman rho = {rho_8:.3f}")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n8-run Spearman rho: {rho_8:.3f} (p={p_8:.4f})")

## Finding 2: Qwen-Opus 相关性是 range-dependent 的

**数据来源**: `d4v7_foundopt_ablation/multi_prompt_validation.json` + `foundopt_all_opus_eval/opus_scores.jsonl`

这是本次实验最反直觉的发现之一:

| 分析范围 | Spearman rho | 含义 |
|---|---|---|
| 跨条件 (3 conditions x 3 tiers) | 高正相关 | Qwen 能区分"好方法 vs 差方法"的大方向 |
| 条件内 (best/median/worst within MAIN) | 低/负相关 | Qwen **不能**在同一方法内区分质量梯度 |
| 8-run 跨方法+跨模型 | 取决于是否包含 235B | 模型能力差异驱动相关 |

**解读**:
- Qwen 作为 grader 能做**粗粒度排序** (A_fresh < 30B-RL < 235B-RL), 这是有用的
- 但在**细粒度**上 (同一 pipeline 的 best vs median), Qwen 给出的分数排序与 Opus 不一致
- 这解释了为什么 RL 优化 Qwen reward 没有在 Opus 上产生改善: RL 追逐的是 Qwen 的噪声, 不是真实质量梯度
- **实际含义**: Qwen reward 适合做**筛选** (reject clearly bad), 不适合做**精细优化** (find the best)

In [ ]:
# ── Cell 9: 多 prompt 验证 — standard vs strict rubric ────────────────────────
# 数据来源: multi_prompt_validation.json
# 每个 plan 有 5 个 signal 的 standard/strict/min 三种评分

print(f"数据来源: {mpv_path}")
print(f"样本数: {len(df_mpv)} plans")
print("Opus 评分标准: 1-10 per dimension, total /40\n")

# 提取 5 个 depth signal 的 std/strict 对比
signals = ["G12_formalism", "G11_evidence_rigor", "G6_reasoning_depth",
           "G4_focus", "G13_risk_awareness"]

print("=== Per-signal: Standard vs Strict rubric 与 Opus 的 Spearman 相关性 ===\n")

corr_rows = []
for sig in signals:
    std_col = f"{sig}_std"
    strict_col = f"{sig}_strict"

    if std_col in df_mpv.columns and strict_col in df_mpv.columns:
        r_std, p_std = spearmanr(df_mpv[std_col], df_mpv["opus"])
        r_strict, p_strict = spearmanr(df_mpv[strict_col], df_mpv["opus"])
        corr_rows.append({
            "Signal": sig,
            "rho (standard)": f"{r_std:.3f}",
            "rho (strict)": f"{r_strict:.3f}",
            "Standard better?": "Yes" if abs(r_std) > abs(r_strict) else "No",
        })

df_corr = pd.DataFrame(corr_rows)
display(df_corr)

# 聚合: depth_agg 对比
print("\n=== Depth aggregate 与 Opus (/40) 相关性 ===")
for mode in ["std", "strict", "min"]:
    col = f"depth_agg_{mode}"
    if col in df_mpv.columns:
        r, p = spearmanr(df_mpv[col], df_mpv["opus"])
        print(f"  depth_agg_{mode}: rho={r:.3f} (p={p:.4f})")

# 可视化: 每个 signal 在 std vs strict 下的均值
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(signals))
width = 0.35

std_means = [df_mpv[f"{s}_std"].mean() for s in signals]
strict_means = [df_mpv[f"{s}_strict"].mean() for s in signals]

ax.bar(x - width/2, std_means, width, label="Standard rubric", color="#4CAF50", alpha=0.8)
ax.bar(x + width/2, strict_means, width, label="Strict rubric", color="#F44336", alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", "\n") for s in signals], fontsize=8)
ax.set_ylabel("Mean Score (1-5)")
ax.set_title("Standard vs Strict Rubric: Mean Signal Scores Across 9 Plans")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\n结论: Strict rubric 导致几乎所有 signal 压缩到 1, 丧失区分度。")
print("Standard rubric 保留了更多信息, 与 Opus (/40) 相关性更高。")

## Finding 3: Standard Rubric 已是最优, Strict 变体全部更差

**数据来源**: `d4v7_foundopt_ablation/multi_prompt_validation.json`

实验设计: 对 9 个 plans (3 conditions x 3 quality tiers), 用 5 个 depth signals 的 3 种 rubric 变体 (standard, strict, min) 分别评分, 然后计算与 Opus ground truth 的相关性。

**结果**:
- Standard rubric 在几乎所有 signal 上都与 Opus 有更高的 Spearman rho
- Strict rubric 把绝大多数分数压到 1/5, 丧失了区分度 (信息论: 熵趋近于 0)
- Min rubric 更极端, 几乎完全退化

**含义**:
- "让 grader 更严格" 并不等于 "让 grader 更准确"
- 当前 standard rubric 已经在 Qwen3-30B 的能力范围内达到了最优的 Opus 相关性
- 进一步改善需要的不是 prompt engineering, 而是更强的 grader model (即 Finding 1 的结论)

In [ ]:
# ── Cell 11: 多目标 B4 结果 ──────────────────────────────────────────────────
# 数据来源: wave1_opus_eval/opus_scores.jsonl + wave2_opus_eval/opus_scores.jsonl
# 6 个 goal x (generated + reference)

wave2_path = ANALYSIS / "wave2_opus_eval/opus_scores.jsonl"
wave2 = []
with open(wave2_path) as f:
    for line in f:
        wave2.append(json.loads(line))
df_wave2 = pd.DataFrame(wave2)

df_all_goals = pd.concat([df_wave1, df_wave2], ignore_index=True)

print(f"数据来源: {wave1_path} + {wave2_path}")
print(f"样本数: {len(df_all_goals)} rows ({len(df_all_goals)//2} goals x 2 types)")
print("评分标准: 1-10 per dimension, total /40\n")

# 表格
print("=== 多目标 B4 Opus 评估 ===\n")
display(df_all_goals[["goal", "type", "depth", "methods", "feasibility", "grounding", "total"]]
        .sort_values(["goal", "type"]).reset_index(drop=True))

# generated vs reference 汇总统计
gen = df_all_goals[df_all_goals["type"] == "generated"]
ref = df_all_goals[df_all_goals["type"] == "reference"]

print(f"\nGenerated 平均 Opus: {gen['total'].mean():.1f}/40 (range: {gen['total'].min()}-{gen['total'].max()})")
print(f"Reference 平均 Opus: {ref['total'].mean():.1f}/40 (range: {ref['total'].min()}-{ref['total'].max()})")
print(f"平均差距: {ref['total'].mean() - gen['total'].mean():.1f} points")

# ── 柱状图: 6 goals x gen vs ref ─────────────────────────────────────────────
goals = df_all_goals["goal"].unique()
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(goals))
width = 0.35

gen_totals = [gen[gen["goal"] == g]["total"].iloc[0] for g in goals]
ref_totals = [ref[ref["goal"] == g]["total"].iloc[0] for g in goals]

bars1 = ax.bar(x - width/2, gen_totals, width, label="Generated (B4, 30B)",
               color="#FF9800", edgecolor="white")
bars2 = ax.bar(x + width/2, ref_totals, width, label="Reference (human expert)",
               color="#4CAF50", edgecolor="white")

# 标注
for i, (gv, rv) in enumerate(zip(gen_totals, ref_totals)):
    ax.text(i - width/2, gv + 0.3, str(gv), ha="center", fontweight="bold", fontsize=10)
    ax.text(i + width/2, rv + 0.3, str(rv), ha="center", fontweight="bold", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels([g.replace("_", "\n") for g in goals], fontsize=9)
ax.set_ylabel("Opus Total Score (/40)")
ax.set_title("Multi-Goal B4 (30B): Generated vs Reference Plans (1-10 scale)")
ax.legend(fontsize=10)
ax.set_ylim(0, 44)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# ── 分维度对比 ───────────────────────────────────────────────────────────────
dims = ["depth", "methods", "feasibility", "grounding"]
print("\n=== 分维度 Generated 平均 vs Reference 平均 ===")
for d in dims:
    g_mean = gen[d].mean()
    r_mean = ref[d].mean()
    print(f"  {d:15s}: Gen {g_mean:.1f} / Ref {r_mean:.1f}  (gap: {r_mean - g_mean:.1f})")

## Finding 4-8: 额外发现汇总

### Finding 4: GAPO Diversity RL 防止坍缩但不提升 Opus

**数据来源**: `foundopt_all_opus_eval/opus_scores.jsonl` (tag: GAPO)

- GAPO (Generalized Advantage Policy Optimization + NN-distance diversity bonus) 设计目标是防止 RL 训练时的模板坍缩
- 结果: GAPO 的 Opus = 17/40, 与 B4_orig (15/40) 和 MAIN_orig (14/40) 在同一量级
- GAPO 确实保持了生成多样性 (diversity_method=gapo, Qwen=1.000), 但多样性本身不等于质量
- **结论**: diversity bonus 解决了一个真实问题 (坍缩), 但不是 Opus 质量的瓶颈

### Finding 5: RL 梯度对共享 LoRA 的权重污染

**数据来源**: `d4v7_foundopt_ablation/data/opus_eval_results.jsonl` (A_fresh runs)

- A_fresh (仅在 fresh generation 上做 RL): Opus best=11, median=11, worst=7
- MAIN (full RL): Opus best=16, median=15, worst=7
- A_fresh 的 Opus ceiling (11) 低于 B4 (20), 说明 RL 梯度更新反而**降低了** B4 搜索的质量
- 共享 LoRA 上的 RL 更新会污染 policy 的基础生成能力

### Finding 6: B4 本质上是搜索算法

**数据来源**: configs (`skip_rl_update=true`) + `foundopt_all_opus_eval/opus_scores.jsonl`

- B4 不做任何梯度更新, 仅依赖: (1) 生成多个候选 (2) grader 打分 (3) buffer 保留最佳 (4) critique-revise 改进
- B4_orig (25 iters, grader_repeats=2): Opus 15/40
- B4_paper (10 iters, grader_repeats=1): Opus 19/40
- **B4 的性能不随迭代数线性增长** — 更多迭代没有更好的结果, 因为 grader 天花板在那里

### Finding 7: 跨领域一致性 (~19 point gap)

**数据来源**: `wave1_opus_eval + wave2_opus_eval` (6 goals)

- 6 个不同领域的 goal (foundopt, causal healthcare, chemo toxicity, ecosystem dynamics, sentencing disparities, climate displacement)
- Generated plans: 14-18/40, Reference: 34-37/40
- 差距在 ~19 points, **跨领域高度一致**
- 这不是某个特定 goal 的问题, 而是模型能力的系统性上界

### Finding 8: 模型能力是天花板

**数据来源**: 所有数据源的交叉验证

核心逻辑链:
1. 30B grader 的 Qwen reward 与 Opus 在细粒度上不相关 (Finding 2)
2. 更严格的 rubric 不能改善这个相关性 (Finding 3)
3. 235B policy+grader 直接把 Opus 从 13-19 提到 26-29 (Finding 1)
4. 跨领域差距恒定 ~19 points (Finding 7)

**结论**: 瓶颈不在 RL 算法、不在 rubric 设计、不在训练配方。瓶颈在模型自身的知识和推理深度。

## 修正后的认知框架

| 维度 | 旧认知 | 修正后认知 | 依据 |
|---|---|---|---|
| **RL 的价值** | RL 梯度更新能持续提升计划质量 | RL 在 Qwen reward 饱和区只追逐噪声; 真实价值仅在探索 (BoN=1 fresh) | Finding 1, 2 |
| **Grader 改进** | 更严格的 rubric prompt = 更好的 grader | Strict rubric 丧失区分度, standard 已是 30B 能力下的最优 | Finding 3 |
| **B4 vs MAIN** | B4 是 baseline, MAIN 应该更好 | B4 >= MAIN 在 Opus 上; RL 梯度可能反而污染权重 | Finding 5, 6 |
| **Goodhart** | Goodhart 是特定信号的问题 | Goodhart 是 range-dependent: Qwen 能粗排但不能精排 | Finding 2 |
| **模型升级** | 30B -> 235B 是渐进改善 | 30B -> 235B 是质变 (+12 Opus points, 27.5 vs 15.5/40), 远超任何 RL 技巧 | Finding 1 |
| **跨领域泛化** | 不同 goal 可能表现差异大 | ~19 point gap 跨领域稳定 (gen 16.2 vs ref 35.2/40) — 系统性瓶颈, 非偶发 | Finding 7 |
| **GAPO/diversity** | Diversity bonus 能带来质量提升 | Diversity 防止坍缩但不提升质量 — 解决了错误的问题 | Finding 4 |

**一句话总结**: 我们试图用 RL 算法的复杂度来弥补 grader model 能力的不足。正确的方向是升级模型, 而非优化算法。

## Next Steps

### 优先级 1: 切换到 235B policy + grader
- 已验证 235B 带来 +12 Opus 的质变 (BR_MAIN_235B: 26/40, BR_B4_235B: 29/40)
- 235B 的 BR_B4 已达 29/40, 距 reference (34-37/40) 仅 5-8 points
- 下一步: 在 235B 上跑完整的 B4 pipeline (当前 BR 只完成了 ~30 iters)

### 优先级 2: Wave 2 已完成
- Wave 1 已完成 4 个 STEM 目标 (foundopt, causal_healthcare, chemo_toxicity, ecosystem_dynamics)
- Wave 2 已完成: sentencing_disparities (Opus 17/40), climate_displacement (Opus 14/40)
- 验证 ~19 point gap 在非 STEM 领域是否依然成立

### 优先级 3: Full Opus eval across 6+ goals
- 当前 multi-goal eval 仅覆盖 B4 (30B)
- 需要: 235B B4 在多目标上的 Opus eval
- 目标: 确认 235B 的跨领域泛化性

### 长期方向
- 如果 235B B4 在 6 goals 上稳定 26-29/40, 剩余的 8-11 points gap 可能需要:
  - 更强的 policy model (e.g., 72B dense 或更大)
  - RAG-augmented generation (引入真实文献)
  - Multi-turn refinement with stronger judge in the loop